# Purpose

The purpose of this notebook is to show that the code works as intended and the results are reproducible. This is done by randomly selecting some simulation $i \in [0,999]$ and a random parameter from the parameter set under $\textrm{seed} = 123$. The output is checked against the saved results corresponding to $i$ and the parameter value.

In [1]:
import pandas as pd
import numpy as np
import scipy
import scipy.sparse as sp
import scipy.io as sio
import scipy.stats as stats
from tqdm.notebook import tqdm


import os
os.environ["R_HOME"] = f"{os.environ['CONDA_PREFIX']}\\Lib\\R"


from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

from plotnine import *

import matplotlib.pyplot as plt 

import pickle

from joblib import Parallel, delayed

import sys
from pathlib import Path

# Get project root as parent of notebooks/
PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.methods.GWASH_funcs import *
from src.methods.GWASH_sim_funcs import *
from src.methods.ldsc_barebones import *

from src.simtools.sim_utils import *
from src.simtools.data_generation import data_generation as data_generation
from src.simtools.data_preprocessing import data_preprocessing as data_preprocessing
from src.simtools.do_analysis import do_analysis as do_analysis
from src.simtools.run_simulations import run_simulations as run_simulations
from src.simtools.data_generation import gen_ref_ldscores_panel as gen_ref_ldscores_panel
from src.simtools.visualization import visualize

from natsort import natsorted

import pickle

seed_num = 123
np.random.seed(seed_num)

os.chdir(PROJECT_ROOT)

# AR1

## $\rho$

### Individual-level

In [2]:


i = np.random.choice(np.arange(0,1000,1),size = 1).item()
rhos = [0,0.4,0.8,0.9,0.95,0.99,0.995]
rho = np.random.choice(rhos,size = 1).item()


n,m = 5000,10000


X_properties = {'n': n,'m':m,'pm_causal':None,'fixed_m_causal': False,'sigma_s':0, 'rho1':0.995,'rho2':None,'Fst':0}
ref_X_properties = {'ref_n': n,'ref_pm_causal':None,'ref_fixed_m_causal': False,'ref_sigma_s':0, 'ref_rho1':0.995,'ref_rho2':None,'ref_Fst':0}
ld_mat_properties = {'realistic': False,'prefix': None,'make_ref_ldscores':False}
method_properties = {'reml_tol': 1e-8,'reml_max_iters': 100}
stats_properties = {'scaleX': True,'scaley': True, 'nPCs':5,'regress_PC_out':False,'regress_X_on_PC': False,'regress_y_on_PC': False}
simul_properties = {'num_sims':np.nan,'h2_pop': 0.2}
debug_properties = {'calc_mu_hat_2_fast': True,'old':False,'track_progress':True}

list_of_dicts = [X_properties,ref_X_properties,ld_mat_properties,method_properties,stats_properties,simul_properties,debug_properties]
params  = combine_all_dicts(list_of_dicts)

to_run = dict()
to_run['demonstration'] = params

sim_key = 'demonstration'
locals().update(to_run[sim_key])
res_dict = dict()
res_dict_raw = dict()

#rhos = [0.995]
counter = -1
multithreading = True


counter += 1
key = str(rho)

if make_ref_ldscores:
    ref_data_gen = data_generation(n = ref_n,m= m,Fst = Fst,rho1 = rho,rho2 = ref_rho2,sigma_s = ref_sigma_s,h2_pop = h2_pop,pm_causal = pm_causal,fixed_m_causal = ref_fixed_m_causal, old = old,realistic = realistic,prefix = prefix,use_gwash_m = False)
    ref_ldscores,ref_mu2_hat,ref_mu3_hat,X_ref = gen_ref_ldscores_panel(ref_data_gen,seed = seed_num,nPCs = nPCs, regress_X_on_PC = regress_X_on_PC, regress_y_on_PC = regress_y_on_PC)
    n_tilde = ref_data_gen.n
else:
    ref_data_gen = None
    ref_ldscores = None
    n_tilde = None

    ref_mu2_hat = None
    ref_mu3_hat = None
    X_ref = None

n_jobs = 1
    
my_data_gen = data_generation(n = n,m= m,Fst = Fst,rho1 = rho,rho2 = rho2,sigma_s = sigma_s,h2_pop = h2_pop,pm_causal = pm_causal,fixed_m_causal = fixed_m_causal, old = old,realistic = realistic,prefix = prefix,ref_ldscores = ref_ldscores,ref_mu2_hat = ref_mu2_hat,ref_mu3_hat = ref_mu3_hat)
res_dfs = run_simulations(my_data_gen,seed = seed_num,scaleX = scaleX,scaley = scaley,num_sims = num_sims,nPCs = nPCs,regress_PC_out = regress_PC_out,regress_X_on_PC = regress_X_on_PC,regress_y_on_PC = regress_y_on_PC, multithreading = multithreading, n_jobs = n_jobs, realistic = realistic, calc_mu_hat_2_fast = calc_mu_hat_2_fast,reml_tol = reml_tol,track_progress = track_progress).run_single_simulation_publication(i=i)

res_dfs

INFO:root:Starting simulation 510
INFO:root:Finished simulation 510


,h2_gcta,h2_gwash,h2_ldsc_reg,icpt_ldsc_reg,h2_ldsc_fixed,h2_samp,h2_gwash_sample_theoretical_se,h2_ldsc_reg_jackknife_se,h2_ldsc_fixed_jackknife_se
0,0.18698,0.227083,0.215902,1.583025,0.225124,0.223282,0.016108,0.271622,0.024733


In [3]:
file = open('save_data/supplementary/AR1/single_param/rho.pkl','rb')
saved_data = pickle.load(file)

saved_data_at_i = saved_data[str(rho)].iloc[[i]].reset_index(drop = True)

In [4]:
res_dfs - saved_data_at_i

,h2_gcta,h2_gwash,h2_ldsc_reg,icpt_ldsc_reg,h2_ldsc_fixed,h2_samp,h2_gwash_sample_theoretical_se,h2_ldsc_reg_jackknife_se,h2_ldsc_fixed_jackknife_se
0,5.551115e-16,5.551115e-17,5.511425e-13,-2.739164e-11,-5.356826e-15,0.0,1.447457e-08,6.000755e-14,1.387779e-17


### Reference Panel

In [ ]:
seed_num = 123

i = np.random.choice(np.arange(0,1000,1),size = 1).item()
rhos = [0,0.4,0.8,0.9,0.95,0.99,0.995]
rho = np.random.choice(rhos,size = 1).item()


n,m = 5000,10000


X_properties = {'n': n,'m':m,'pm_causal':None,'fixed_m_causal': False,'sigma_s':0, 'rho1':0.995,'rho2':None,'Fst':0}
ref_X_properties = {'ref_n': n,'ref_pm_causal':None,'ref_fixed_m_causal': False,'ref_sigma_s':0, 'ref_rho1':0.995,'ref_rho2':None,'ref_Fst':0}
ld_mat_properties = {'realistic': False,'prefix': None,'make_ref_ldscores':True}
method_properties = {'reml_tol': 1e-8,'reml_max_iters': 100}
stats_properties = {'scaleX': True,'scaley': True, 'nPCs':5,'regress_PC_out':False,'regress_X_on_PC': False,'regress_y_on_PC': False}
simul_properties = {'num_sims':np.nan,'h2_pop': 0.2}
debug_properties = {'calc_mu_hat_2_fast': True,'old':False,'track_progress':True}

list_of_dicts = [X_properties,ref_X_properties,ld_mat_properties,method_properties,stats_properties,simul_properties,debug_properties]
params  = combine_all_dicts(list_of_dicts)

to_run = dict()
to_run['demonstration'] = params

sim_key = 'demonstration'
locals().update(to_run[sim_key])
res_dict = dict()
res_dict_raw = dict()

#rhos = [0.995]
counter = -1
multithreading = True


counter += 1
key = str(rho)

if make_ref_ldscores:
    ref_data_gen = data_generation(n = ref_n,m= m,Fst = Fst,rho1 = rho,rho2 = ref_rho2,sigma_s = ref_sigma_s,h2_pop = h2_pop,pm_causal = pm_causal,fixed_m_causal = ref_fixed_m_causal, old = old,realistic = realistic,prefix = prefix,use_gwash_m = False)
    ref_ldscores,ref_mu2_hat,ref_mu3_hat,X_ref = gen_ref_ldscores_panel(ref_data_gen,seed = seed_num,nPCs = nPCs, regress_X_on_PC = regress_X_on_PC, regress_y_on_PC = regress_y_on_PC)
    n_tilde = ref_data_gen.n
else:
    ref_data_gen = None
    ref_ldscores = None
    n_tilde = None

    ref_mu2_hat = None
    ref_mu3_hat = None
    X_ref = None

n_jobs = 1
    
my_data_gen = data_generation(n = n,m= m,Fst = Fst,rho1 = rho,rho2 = rho2,sigma_s = sigma_s,h2_pop = h2_pop,pm_causal = pm_causal,fixed_m_causal = fixed_m_causal, old = old,realistic = realistic,prefix = prefix,ref_ldscores = ref_ldscores,ref_mu2_hat = ref_mu2_hat,ref_mu3_hat = ref_mu3_hat)
res_dfs = run_simulations(my_data_gen,seed = seed_num,scaleX = scaleX,scaley = scaley,num_sims = num_sims,nPCs = nPCs,regress_PC_out = regress_PC_out,regress_X_on_PC = regress_X_on_PC,regress_y_on_PC = regress_y_on_PC, multithreading = multithreading, n_jobs = n_jobs, realistic = realistic, calc_mu_hat_2_fast = calc_mu_hat_2_fast,reml_tol = reml_tol,track_progress = track_progress).run_single_simulation_publication(i=i)

res_dfs

In [ ]:
file = open('save_data/main_text/AR1/single_param/rho.pkl','rb')
saved_data = pickle.load(file)

saved_data_at_i = saved_data[str(rho)].iloc[[i]].reset_index(drop = True)

In [ ]:
res_dfs - saved_data_at_i

## $\mathrm{prop_{causal}}$

### Individual-level

In [ ]:
seed_num = 123

i = np.random.choice(np.arange(0,1000,1),size = 1).item()



n,m = 5000,10000


X_properties = {'n': n,'m':m,'pm_causal':None,'fixed_m_causal': False,'sigma_s':0, 'rho1':0.995,'rho2':None,'Fst':0}
ref_X_properties = {'ref_n': n,'ref_pm_causal':None,'ref_fixed_m_causal': False,'ref_sigma_s':0, 'ref_rho1':0.995,'ref_rho2':None,'ref_Fst':0}
ld_mat_properties = {'realistic': False,'prefix': None,'make_ref_ldscores':False}
method_properties = {'reml_tol': 1e-8,'reml_max_iters': 100}
stats_properties = {'scaleX': True,'scaley': True, 'nPCs':5,'regress_PC_out':False,'regress_X_on_PC': False,'regress_y_on_PC': False}
simul_properties = {'num_sims':np.nan,'h2_pop': 0.2}
debug_properties = {'calc_mu_hat_2_fast': True,'old':False,'track_progress':True}

list_of_dicts = [X_properties,ref_X_properties,ld_mat_properties,method_properties,stats_properties,simul_properties,debug_properties]
params  = combine_all_dicts(list_of_dicts)

to_run = dict()
to_run['demonstration'] = params

sim_key = 'demonstration'
locals().update(to_run[sim_key])
res_dict = dict()
res_dict_raw = dict()

#rhos = [0.995]
counter = -1
multithreading = True


counter += 1
pm_causals = [0.005,0.05,0.5,0.75,float(1)]
pm_causal = np.random.choice(pm_causals,size = 1).item()

if make_ref_ldscores:
    ref_data_gen = data_generation(n = ref_n,m= m,Fst = Fst,rho1 = ref_rho1,rho2 = ref_rho2,sigma_s = ref_sigma_s,h2_pop = h2_pop,pm_causal = pm_causal,fixed_m_causal = ref_fixed_m_causal, old = old,realistic = realistic,prefix = prefix,use_gwash_m = False)
    ref_ldscores,ref_mu2_hat,ref_mu3_hat,X_ref = gen_ref_ldscores_panel(ref_data_gen,seed = seed_num,nPCs = nPCs, regress_X_on_PC = regress_X_on_PC, regress_y_on_PC = regress_y_on_PC)
    n_tilde = ref_data_gen.n
else:
    ref_data_gen = None
    ref_ldscores = None
    n_tilde = None

    ref_mu2_hat = None
    ref_mu3_hat = None
    X_ref = None

n_jobs = 1
    
my_data_gen = data_generation(n = n,m= m,Fst = Fst,rho1 = rho1,rho2 = rho2,sigma_s = sigma_s,h2_pop = h2_pop,pm_causal = pm_causal,fixed_m_causal = fixed_m_causal, old = old,realistic = realistic,prefix = prefix,ref_ldscores = ref_ldscores,ref_mu2_hat = ref_mu2_hat,ref_mu3_hat = ref_mu3_hat)
res_dfs = run_simulations(my_data_gen,seed = seed_num,scaleX = scaleX,scaley = scaley,num_sims = num_sims,nPCs = nPCs,regress_PC_out = regress_PC_out,regress_X_on_PC = regress_X_on_PC,regress_y_on_PC = regress_y_on_PC, multithreading = multithreading, n_jobs = n_jobs, realistic = realistic, calc_mu_hat_2_fast = calc_mu_hat_2_fast,reml_tol = reml_tol,track_progress = track_progress).run_single_simulation_publication(i=i)

res_dfs

In [ ]:
file = open('save_data/supplementary/AR1/single_param/pm_causal.pkl','rb')
saved_data = pickle.load(file)

saved_data_at_i = saved_data[str(pm_causal)].iloc[[i]].reset_index(drop = True)

In [ ]:
res_dfs - saved_data_at_i

### Reference Panel

In [ ]:
seed_num = 123

i = np.random.choice(np.arange(0,1000,1),size = 1).item()



n,m = 5000,10000


X_properties = {'n': n,'m':m,'pm_causal':None,'fixed_m_causal': False,'sigma_s':0, 'rho1':0.995,'rho2':None,'Fst':0}
ref_X_properties = {'ref_n': n,'ref_pm_causal':None,'ref_fixed_m_causal': False,'ref_sigma_s':0, 'ref_rho1':0.995,'ref_rho2':None,'ref_Fst':0}
ld_mat_properties = {'realistic': False,'prefix': None,'make_ref_ldscores':True}
method_properties = {'reml_tol': 1e-8,'reml_max_iters': 100}
stats_properties = {'scaleX': True,'scaley': True, 'nPCs':5,'regress_PC_out':False,'regress_X_on_PC': False,'regress_y_on_PC': False}
simul_properties = {'num_sims':np.nan,'h2_pop': 0.2}
debug_properties = {'calc_mu_hat_2_fast': True,'old':False,'track_progress':True}

list_of_dicts = [X_properties,ref_X_properties,ld_mat_properties,method_properties,stats_properties,simul_properties,debug_properties]
params  = combine_all_dicts(list_of_dicts)

to_run = dict()
to_run['demonstration'] = params

sim_key = 'demonstration'
locals().update(to_run[sim_key])
res_dict = dict()
res_dict_raw = dict()

#rhos = [0.995]
counter = -1
multithreading = True


counter += 1
pm_causals = [0.005,0.05,0.5,0.75,float(1)]
pm_causal = np.random.choice(pm_causals,size = 1).item()

if make_ref_ldscores:
    ref_data_gen = data_generation(n = ref_n,m= m,Fst = Fst,rho1 = ref_rho1,rho2 = ref_rho2,sigma_s = ref_sigma_s,h2_pop = h2_pop,pm_causal = pm_causal,fixed_m_causal = ref_fixed_m_causal, old = old,realistic = realistic,prefix = prefix,use_gwash_m = False)
    ref_ldscores,ref_mu2_hat,ref_mu3_hat,X_ref = gen_ref_ldscores_panel(ref_data_gen,seed = seed_num,nPCs = nPCs, regress_X_on_PC = regress_X_on_PC, regress_y_on_PC = regress_y_on_PC)
    n_tilde = ref_data_gen.n
else:
    ref_data_gen = None
    ref_ldscores = None
    n_tilde = None

    ref_mu2_hat = None
    ref_mu3_hat = None
    X_ref = None

n_jobs = 1
    
my_data_gen = data_generation(n = n,m= m,Fst = Fst,rho1 = rho1,rho2 = rho2,sigma_s = sigma_s,h2_pop = h2_pop,pm_causal = pm_causal,fixed_m_causal = fixed_m_causal, old = old,realistic = realistic,prefix = prefix,ref_ldscores = ref_ldscores,ref_mu2_hat = ref_mu2_hat,ref_mu3_hat = ref_mu3_hat)
res_dfs = run_simulations(my_data_gen,seed = seed_num,scaleX = scaleX,scaley = scaley,num_sims = num_sims,nPCs = nPCs,regress_PC_out = regress_PC_out,regress_X_on_PC = regress_X_on_PC,regress_y_on_PC = regress_y_on_PC, multithreading = multithreading, n_jobs = n_jobs, realistic = realistic, calc_mu_hat_2_fast = calc_mu_hat_2_fast,reml_tol = reml_tol,track_progress = track_progress).run_single_simulation_publication(i=i)

res_dfs

In [ ]:
file = open('save_data/main_text/AR1/single_param/pm_causal.pkl','rb')
saved_data = pickle.load(file)

saved_data_at_i = saved_data[str(pm_causal)].iloc[[i]].reset_index(drop = True)

In [ ]:
res_dfs - saved_data_at_i

## $\sigma_{s}$

### Individual-level

In [ ]:
seed_num = 123

i = np.random.choice(np.arange(0,1000,1),size = 1).item()



n,m = 5000,10000


X_properties = {'n': n,'m':m,'pm_causal':None,'fixed_m_causal': False,'sigma_s':0, 'rho1':0.995,'rho2':None,'Fst':0}
ref_X_properties = {'ref_n': n,'ref_pm_causal':None,'ref_fixed_m_causal': False,'ref_sigma_s':0, 'ref_rho1':0.995,'ref_rho2':None,'ref_Fst':0}
ld_mat_properties = {'realistic': False,'prefix': None,'make_ref_ldscores':False}
method_properties = {'reml_tol': 1e-8,'reml_max_iters': 100}
stats_properties = {'scaleX': True,'scaley': True, 'nPCs':5,'regress_PC_out':False,'regress_X_on_PC': False,'regress_y_on_PC': False}
simul_properties = {'num_sims':np.nan,'h2_pop': 0.2}
debug_properties = {'calc_mu_hat_2_fast': True,'old':False,'track_progress':True}

list_of_dicts = [X_properties,ref_X_properties,ld_mat_properties,method_properties,stats_properties,simul_properties,debug_properties]
params  = combine_all_dicts(list_of_dicts)

to_run = dict()
to_run['demonstration'] = params

sim_key = 'demonstration'
locals().update(to_run[sim_key])
res_dict = dict()
res_dict_raw = dict()

#rhos = [0.995]
counter = -1
multithreading = True


counter += 1
sigma_ss = [0,0.2,0.4,0.6,0.8]
sigma_s = np.random.choice(sigma_ss,size = 1).item()

if make_ref_ldscores:
    ref_data_gen = data_generation(n = ref_n,m= m,Fst = Fst,rho1 = ref_rho1,rho2 = ref_rho2,sigma_s = ref_sigma_s,h2_pop = h2_pop,pm_causal = pm_causal,fixed_m_causal = ref_fixed_m_causal, old = old,realistic = realistic,prefix = prefix,use_gwash_m = False)
    ref_ldscores,ref_mu2_hat,ref_mu3_hat,X_ref = gen_ref_ldscores_panel(ref_data_gen,seed = seed_num,nPCs = nPCs, regress_X_on_PC = regress_X_on_PC, regress_y_on_PC = regress_y_on_PC)
    n_tilde = ref_data_gen.n
else:
    ref_data_gen = None
    ref_ldscores = None
    n_tilde = None

    ref_mu2_hat = None
    ref_mu3_hat = None
    X_ref = None

n_jobs = 1
    
my_data_gen = data_generation(n = n,m= m,Fst = Fst,rho1 = rho1,rho2 = rho2,sigma_s = sigma_s,h2_pop = h2_pop,pm_causal = pm_causal,fixed_m_causal = fixed_m_causal, old = old,realistic = realistic,prefix = prefix,ref_ldscores = ref_ldscores,ref_mu2_hat = ref_mu2_hat,ref_mu3_hat = ref_mu3_hat)
res_dfs = run_simulations(my_data_gen,seed = seed_num,scaleX = scaleX,scaley = scaley,num_sims = num_sims,nPCs = nPCs,regress_PC_out = regress_PC_out,regress_X_on_PC = regress_X_on_PC,regress_y_on_PC = regress_y_on_PC, multithreading = multithreading, n_jobs = n_jobs, realistic = realistic, calc_mu_hat_2_fast = calc_mu_hat_2_fast,reml_tol = reml_tol,track_progress = track_progress).run_single_simulation_publication(i=i)

res_dfs

In [ ]:
file = open('save_data/supplementary/AR1/single_param/sigma_s.pkl','rb')
saved_data = pickle.load(file)

if sigma_s == 0:
    sigma_s = int(sigma_s)

saved_data_at_i = saved_data[str(sigma_s)].iloc[[i]].reset_index(drop = True)

In [ ]:
res_dfs - saved_data_at_i

### Reference Panel

In [ ]:
seed_num = 123

i = np.random.choice(np.arange(0,1000,1),size = 1).item()



n,m = 5000,10000


X_properties = {'n': n,'m':m,'pm_causal':None,'fixed_m_causal': False,'sigma_s':0, 'rho1':0.995,'rho2':None,'Fst':0}
ref_X_properties = {'ref_n': n,'ref_pm_causal':None,'ref_fixed_m_causal': False,'ref_sigma_s':0, 'ref_rho1':0.995,'ref_rho2':None,'ref_Fst':0}
ld_mat_properties = {'realistic': False,'prefix': None,'make_ref_ldscores':True}
method_properties = {'reml_tol': 1e-8,'reml_max_iters': 100}
stats_properties = {'scaleX': True,'scaley': True, 'nPCs':5,'regress_PC_out':False,'regress_X_on_PC': False,'regress_y_on_PC': False}
simul_properties = {'num_sims':np.nan,'h2_pop': 0.2}
debug_properties = {'calc_mu_hat_2_fast': True,'old':False,'track_progress':True}

list_of_dicts = [X_properties,ref_X_properties,ld_mat_properties,method_properties,stats_properties,simul_properties,debug_properties]
params  = combine_all_dicts(list_of_dicts)

to_run = dict()
to_run['demonstration'] = params

sim_key = 'demonstration'
locals().update(to_run[sim_key])
res_dict = dict()
res_dict_raw = dict()

#rhos = [0.995]
counter = -1
multithreading = True


counter += 1
sigma_ss = [0,0.2,0.4,0.6,0.8]
sigma_s = np.random.choice(sigma_ss,size = 1).item()

if make_ref_ldscores:
    ref_data_gen = data_generation(n = ref_n,m= m,Fst = Fst,rho1 = ref_rho1,rho2 = ref_rho2,sigma_s = sigma_s,h2_pop = h2_pop,pm_causal = pm_causal,fixed_m_causal = ref_fixed_m_causal, old = old,realistic = realistic,prefix = prefix,use_gwash_m = False)
    ref_ldscores,ref_mu2_hat,ref_mu3_hat,X_ref = gen_ref_ldscores_panel(ref_data_gen,seed = seed_num,nPCs = nPCs, regress_X_on_PC = regress_X_on_PC, regress_y_on_PC = regress_y_on_PC)
    n_tilde = ref_data_gen.n
else:
    ref_data_gen = None
    ref_ldscores = None
    n_tilde = None

    ref_mu2_hat = None
    ref_mu3_hat = None
    X_ref = None

n_jobs = 1
    
my_data_gen = data_generation(n = n,m= m,Fst = Fst,rho1 = rho1,rho2 = rho2,sigma_s = sigma_s,h2_pop = h2_pop,pm_causal = pm_causal,fixed_m_causal = fixed_m_causal, old = old,realistic = realistic,prefix = prefix,ref_ldscores = ref_ldscores,ref_mu2_hat = ref_mu2_hat,ref_mu3_hat = ref_mu3_hat)
res_dfs = run_simulations(my_data_gen,seed = seed_num,scaleX = scaleX,scaley = scaley,num_sims = num_sims,nPCs = nPCs,regress_PC_out = regress_PC_out,regress_X_on_PC = regress_X_on_PC,regress_y_on_PC = regress_y_on_PC, multithreading = multithreading, n_jobs = n_jobs, realistic = realistic, calc_mu_hat_2_fast = calc_mu_hat_2_fast,reml_tol = reml_tol,track_progress = track_progress).run_single_simulation_publication(i=i)

res_dfs

In [ ]:
file = open('save_data/main_text/AR1/single_param/sigma_s.pkl','rb')
saved_data = pickle.load(file)

if sigma_s == 0:
    sigma_s = int(sigma_s)

saved_data_at_i = saved_data[str(sigma_s)].iloc[[i]].reset_index(drop = True)

In [ ]:
res_dfs - saved_data_at_i

## $\mathrm{F_{st}}$

### Individual-level

In [ ]:
seed_num = 123

i = np.random.choice(np.arange(0,1000,1),size = 1).item()



n,m = 5000,10000


X_properties = {'n': n,'m':m,'pm_causal':None,'fixed_m_causal': False,'sigma_s':0, 'rho1':0.995,'rho2':None,'Fst':0}
ref_X_properties = {'ref_n': n,'ref_pm_causal':None,'ref_fixed_m_causal': False,'ref_sigma_s':0, 'ref_rho1':0.995,'ref_rho2':None,'ref_Fst':0}
ld_mat_properties = {'realistic': False,'prefix': None,'make_ref_ldscores':False}
method_properties = {'reml_tol': 1e-8,'reml_max_iters': 100}
stats_properties = {'scaleX': True,'scaley': True, 'nPCs':5,'regress_PC_out':False,'regress_X_on_PC': False,'regress_y_on_PC': False}
simul_properties = {'num_sims':np.nan,'h2_pop': 0.2}
debug_properties = {'calc_mu_hat_2_fast': True,'old':False,'track_progress':True}

list_of_dicts = [X_properties,ref_X_properties,ld_mat_properties,method_properties,stats_properties,simul_properties,debug_properties]
params  = combine_all_dicts(list_of_dicts)

to_run = dict()
to_run['demonstration'] = params

sim_key = 'demonstration'
locals().update(to_run[sim_key])
res_dict = dict()
res_dict_raw = dict()

#rhos = [0.995]
counter = -1
multithreading = True


counter += 1
Fsts = [0,0.01,0.03,0.05,0.07,0.1]
Fst = np.random.choice(Fsts,size = 1).item()

if make_ref_ldscores:
    ref_data_gen = data_generation(n = ref_n,m= m,Fst = Fst,rho1 = ref_rho1,rho2 = ref_rho2,sigma_s = ref_sigma_s,h2_pop = h2_pop,pm_causal = pm_causal,fixed_m_causal = ref_fixed_m_causal, old = old,realistic = realistic,prefix = prefix,use_gwash_m = False)
    ref_ldscores,ref_mu2_hat,ref_mu3_hat,X_ref = gen_ref_ldscores_panel(ref_data_gen,seed = seed_num,nPCs = nPCs, regress_X_on_PC = regress_X_on_PC, regress_y_on_PC = regress_y_on_PC)
    n_tilde = ref_data_gen.n
else:
    ref_data_gen = None
    ref_ldscores = None
    n_tilde = None

    ref_mu2_hat = None
    ref_mu3_hat = None
    X_ref = None

n_jobs = 1
    
my_data_gen = data_generation(n = n,m= m,Fst = Fst,rho1 = rho1,rho2 = rho2,sigma_s = sigma_s,h2_pop = h2_pop,pm_causal = pm_causal,fixed_m_causal = fixed_m_causal, old = old,realistic = realistic,prefix = prefix,ref_ldscores = ref_ldscores,ref_mu2_hat = ref_mu2_hat,ref_mu3_hat = ref_mu3_hat)
res_dfs = run_simulations(my_data_gen,seed = seed_num,scaleX = scaleX,scaley = scaley,num_sims = num_sims,nPCs = nPCs,regress_PC_out = regress_PC_out,regress_X_on_PC = regress_X_on_PC,regress_y_on_PC = regress_y_on_PC, multithreading = multithreading, n_jobs = n_jobs, realistic = realistic, calc_mu_hat_2_fast = calc_mu_hat_2_fast,reml_tol = reml_tol,track_progress = track_progress).run_single_simulation_publication(i=i)

res_dfs

In [ ]:
file = open('save_data/supplementary/AR1/single_param/Fst.pkl','rb')
saved_data = pickle.load(file)

if Fst == 0:
    Fst = int(Fst)

saved_data_at_i = saved_data[str(Fst)].iloc[[i]].reset_index(drop = True)

In [ ]:
res_dfs - saved_data_at_i

### Reference Panel

In [ ]:
seed_num = 123

i = np.random.choice(np.arange(0,1000,1),size = 1).item()



n,m = 5000,10000


X_properties = {'n': n,'m':m,'pm_causal':None,'fixed_m_causal': False,'sigma_s':0, 'rho1':0.995,'rho2':None,'Fst':0}
ref_X_properties = {'ref_n': n,'ref_pm_causal':None,'ref_fixed_m_causal': False,'ref_sigma_s':0, 'ref_rho1':0.995,'ref_rho2':None,'ref_Fst':0}
ld_mat_properties = {'realistic': False,'prefix': None,'make_ref_ldscores':True}
method_properties = {'reml_tol': 1e-8,'reml_max_iters': 100}
stats_properties = {'scaleX': True,'scaley': True, 'nPCs':5,'regress_PC_out':False,'regress_X_on_PC': False,'regress_y_on_PC': False}
simul_properties = {'num_sims':np.nan,'h2_pop': 0.2}
debug_properties = {'calc_mu_hat_2_fast': True,'old':False,'track_progress':True}

list_of_dicts = [X_properties,ref_X_properties,ld_mat_properties,method_properties,stats_properties,simul_properties,debug_properties]
params  = combine_all_dicts(list_of_dicts)

to_run = dict()
to_run['demonstration'] = params

sim_key = 'demonstration'
locals().update(to_run[sim_key])
res_dict = dict()
res_dict_raw = dict()

#rhos = [0.995]
counter = -1
multithreading = True


counter += 1
Fsts = [0,0.01,0.03,0.05,0.07,0.1]
Fst = np.random.choice(Fsts,size = 1).item()

if make_ref_ldscores:
    ref_data_gen = data_generation(n = ref_n,m= m,Fst = Fst,rho1 = ref_rho1,rho2 = ref_rho2,sigma_s = ref_sigma_s,h2_pop = h2_pop,pm_causal = pm_causal,fixed_m_causal = ref_fixed_m_causal, old = old,realistic = realistic,prefix = prefix,use_gwash_m = False)
    ref_ldscores,ref_mu2_hat,ref_mu3_hat,X_ref = gen_ref_ldscores_panel(ref_data_gen,seed = seed_num,nPCs = nPCs, regress_X_on_PC = regress_X_on_PC, regress_y_on_PC = regress_y_on_PC)
    n_tilde = ref_data_gen.n
else:
    ref_data_gen = None
    ref_ldscores = None
    n_tilde = None

    ref_mu2_hat = None
    ref_mu3_hat = None
    X_ref = None

n_jobs = 1
    
my_data_gen = data_generation(n = n,m= m,Fst = Fst,rho1 = rho1,rho2 = rho2,sigma_s = sigma_s,h2_pop = h2_pop,pm_causal = pm_causal,fixed_m_causal = fixed_m_causal, old = old,realistic = realistic,prefix = prefix,ref_ldscores = ref_ldscores,ref_mu2_hat = ref_mu2_hat,ref_mu3_hat = ref_mu3_hat)
res_dfs = run_simulations(my_data_gen,seed = seed_num,scaleX = scaleX,scaley = scaley,num_sims = num_sims,nPCs = nPCs,regress_PC_out = regress_PC_out,regress_X_on_PC = regress_X_on_PC,regress_y_on_PC = regress_y_on_PC, multithreading = multithreading, n_jobs = n_jobs, realistic = realistic, calc_mu_hat_2_fast = calc_mu_hat_2_fast,reml_tol = reml_tol,track_progress = track_progress).run_single_simulation_publication(i=i)

res_dfs

In [ ]:
file = open('save_data/main_text/AR1/single_param/Fst.pkl','rb')
saved_data = pickle.load(file)

if Fst == 0:
    Fst = int(Fst)

saved_data_at_i = saved_data[str(Fst)].iloc[[i]].reset_index(drop = True)

In [ ]:
res_dfs - saved_data_at_i

### $\mathrm{F_{st}} + \sigma_{s}$

### Individual-level

In [ ]:
seed_num = 123

i = np.random.choice(np.arange(0,1000,1),size = 1).item()



n,m = 5000,10000


X_properties = {'n': n,'m':m,'pm_causal':None,'fixed_m_causal': False,'sigma_s':0, 'rho1':0.995,'rho2':None,'Fst':0}
ref_X_properties = {'ref_n': n,'ref_pm_causal':None,'ref_fixed_m_causal': False,'ref_sigma_s':0, 'ref_rho1':0.995,'ref_rho2':None,'ref_Fst':0}
ld_mat_properties = {'realistic': False,'prefix': None,'make_ref_ldscores':False}
method_properties = {'reml_tol': 1e-8,'reml_max_iters': 100}
stats_properties = {'scaleX': True,'scaley': True, 'nPCs':5,'regress_PC_out':False,'regress_X_on_PC': False,'regress_y_on_PC': False}
simul_properties = {'num_sims':np.nan,'h2_pop': 0.2}
debug_properties = {'calc_mu_hat_2_fast': True,'old':False,'track_progress':True}

list_of_dicts = [X_properties,ref_X_properties,ld_mat_properties,method_properties,stats_properties,simul_properties,debug_properties]
params  = combine_all_dicts(list_of_dicts)

to_run = dict()
to_run['demonstration'] = params

sim_key = 'demonstration'
locals().update(to_run[sim_key])
res_dict = dict()
res_dict_raw = dict()

#rhos = [0.995]
counter = -1
multithreading = True


counter += 1
Fsts = [0.05,0.1]
Fst = np.random.choice(Fsts,size = 1).item()
sigma_ss = [0,0.2,0.4,0.6,0.8]
sigma_s = np.random.choice(sigma_ss,size = 1).item()

if make_ref_ldscores:
    ref_data_gen = data_generation(n = ref_n,m= m,Fst = Fst,rho1 = ref_rho1,rho2 = ref_rho2,sigma_s = sigma_s,h2_pop = h2_pop,pm_causal = pm_causal,fixed_m_causal = ref_fixed_m_causal, old = old,realistic = realistic,prefix = prefix,use_gwash_m = False)
    ref_ldscores,ref_mu2_hat,ref_mu3_hat,X_ref = gen_ref_ldscores_panel(ref_data_gen,seed = seed_num,nPCs = nPCs, regress_X_on_PC = regress_X_on_PC, regress_y_on_PC = regress_y_on_PC)
    n_tilde = ref_data_gen.n
else:
    ref_data_gen = None
    ref_ldscores = None
    n_tilde = None

    ref_mu2_hat = None
    ref_mu3_hat = None
    X_ref = None

n_jobs = 1
    
my_data_gen = data_generation(n = n,m= m,Fst = Fst,rho1 = rho1,rho2 = rho2,sigma_s = sigma_s,h2_pop = h2_pop,pm_causal = pm_causal,fixed_m_causal = fixed_m_causal, old = old,realistic = realistic,prefix = prefix,ref_ldscores = ref_ldscores,ref_mu2_hat = ref_mu2_hat,ref_mu3_hat = ref_mu3_hat)
res_dfs = run_simulations(my_data_gen,seed = seed_num,scaleX = scaleX,scaley = scaley,num_sims = num_sims,nPCs = nPCs,regress_PC_out = regress_PC_out,regress_X_on_PC = regress_X_on_PC,regress_y_on_PC = regress_y_on_PC, multithreading = multithreading, n_jobs = n_jobs, realistic = realistic, calc_mu_hat_2_fast = calc_mu_hat_2_fast,reml_tol = reml_tol,track_progress = track_progress).run_single_simulation_publication(i=i)

res_dfs

In [ ]:
file = open('save_data/supplementary/AR1/double_param/Fst'+str(Fst).replace('.','')+'sigma_s.pkl','rb')
saved_data = pickle.load(file)

if sigma_s == 0:
    sigma_s = int(sigma_s)

saved_data_at_i = saved_data[str(sigma_s)].iloc[[i]].reset_index(drop = True)

In [ ]:
res_dfs - saved_data_at_i

### Reference Panel

In [ ]:
seed_num = 123

i = np.random.choice(np.arange(0,1000,1),size = 1).item()



n,m = 5000,10000


X_properties = {'n': n,'m':m,'pm_causal':None,'fixed_m_causal': False,'sigma_s':0, 'rho1':0.995,'rho2':None,'Fst':0}
ref_X_properties = {'ref_n': n,'ref_pm_causal':None,'ref_fixed_m_causal': False,'ref_sigma_s':0, 'ref_rho1':0.995,'ref_rho2':None,'ref_Fst':0}
ld_mat_properties = {'realistic': False,'prefix': None,'make_ref_ldscores':True}
method_properties = {'reml_tol': 1e-8,'reml_max_iters': 100}
stats_properties = {'scaleX': True,'scaley': True, 'nPCs':5,'regress_PC_out':False,'regress_X_on_PC': False,'regress_y_on_PC': False}
simul_properties = {'num_sims':np.nan,'h2_pop': 0.2}
debug_properties = {'calc_mu_hat_2_fast': True,'old':False,'track_progress':True}

list_of_dicts = [X_properties,ref_X_properties,ld_mat_properties,method_properties,stats_properties,simul_properties,debug_properties]
params  = combine_all_dicts(list_of_dicts)

to_run = dict()
to_run['demonstration'] = params

sim_key = 'demonstration'
locals().update(to_run[sim_key])
res_dict = dict()
res_dict_raw = dict()

#rhos = [0.995]
counter = -1
multithreading = True


counter += 1
Fsts = [0.05,0.1]
Fst = np.random.choice(Fsts,size = 1).item()
sigma_ss = [0,0.2,0.4,0.6,0.8]
sigma_s = np.random.choice(sigma_ss,size = 1).item()

if make_ref_ldscores:
    ref_data_gen = data_generation(n = ref_n,m= m,Fst = Fst,rho1 = ref_rho1,rho2 = ref_rho2,sigma_s = sigma_s,h2_pop = h2_pop,pm_causal = pm_causal,fixed_m_causal = ref_fixed_m_causal, old = old,realistic = realistic,prefix = prefix,use_gwash_m = False)
    ref_ldscores,ref_mu2_hat,ref_mu3_hat,X_ref = gen_ref_ldscores_panel(ref_data_gen,seed = seed_num,nPCs = nPCs, regress_X_on_PC = regress_X_on_PC, regress_y_on_PC = regress_y_on_PC)
    n_tilde = ref_data_gen.n
else:
    ref_data_gen = None
    ref_ldscores = None
    n_tilde = None

    ref_mu2_hat = None
    ref_mu3_hat = None
    X_ref = None

n_jobs = 1
    
my_data_gen = data_generation(n = n,m= m,Fst = Fst,rho1 = rho1,rho2 = rho2,sigma_s = sigma_s,h2_pop = h2_pop,pm_causal = pm_causal,fixed_m_causal = fixed_m_causal, old = old,realistic = realistic,prefix = prefix,ref_ldscores = ref_ldscores,ref_mu2_hat = ref_mu2_hat,ref_mu3_hat = ref_mu3_hat)
res_dfs = run_simulations(my_data_gen,seed = seed_num,scaleX = scaleX,scaley = scaley,num_sims = num_sims,nPCs = nPCs,regress_PC_out = regress_PC_out,regress_X_on_PC = regress_X_on_PC,regress_y_on_PC = regress_y_on_PC, multithreading = multithreading, n_jobs = n_jobs, realistic = realistic, calc_mu_hat_2_fast = calc_mu_hat_2_fast,reml_tol = reml_tol,track_progress = track_progress).run_single_simulation_publication(i=i)

res_dfs

In [ ]:
file = open('save_data/main_text/AR1/double_param/Fst'+str(Fst).replace('.','')+'sigma_s.pkl','rb')
saved_data = pickle.load(file)

if sigma_s == 0:
    sigma_s = int(sigma_s)

saved_data_at_i = saved_data[str(sigma_s)].iloc[[i]].reset_index(drop = True)

In [ ]:
res_dfs - saved_data_at_i

# Realistic

## $\mathrm{prop_{causal}}$

### Individual-data

In [ ]:
seed_num = 123

i = np.random.choice(np.arange(0,1000,1),size = 1).item()



n = 5000


X_properties = {'n': n,'m':10000,'pm_causal':None,'fixed_m_causal': False,'sigma_s':0, 'rho1':0.995,'rho2':None,'Fst':0}
ref_X_properties = {'ref_n': n,'ref_pm_causal':None,'ref_fixed_m_causal': False,'ref_sigma_s':0, 'ref_rho1':0.995,'ref_rho2':None,'ref_Fst':0}
ld_mat_properties = {'realistic': True,'model_Fst_in_realistic':True,'prefix': '1kg_p1_eur_ben/1kg_p1_eur_chr22','make_ref_ldscores':False}
method_properties = {'reml_tol': 1e-8,'reml_max_iters': 100}
stats_properties = {'scaleX': True,'scaley': True, 'nPCs':5,'regress_PC_out':False,'regress_X_on_PC': False,'regress_y_on_PC': False}
simul_properties = {'num_sims':np.nan,'h2_pop': 0.2}
debug_properties = {'calc_mu_hat_2_fast': True,'old':False,'track_progress':True}

list_of_dicts = [X_properties,ref_X_properties,ld_mat_properties,method_properties,stats_properties,simul_properties,debug_properties]
params  = combine_all_dicts(list_of_dicts)

to_run = dict()
to_run['demonstration'] = params

sim_key = 'demonstration'
locals().update(to_run[sim_key])
res_dict = dict()
res_dict_raw = dict()

#rhos = [0.995]
counter = -1
multithreading = True


counter += 1
pm_causals = [0.005,0.05,0.5,0.75,float(1)]
pm_causal = np.random.choice(pm_causals,size = 1).item()

if make_ref_ldscores:
    ref_data_gen = data_generation(n = ref_n,m= m,Fst = Fst,rho1 = ref_rho1,rho2 = ref_rho2,sigma_s = ref_sigma_s,h2_pop = h2_pop,pm_causal = pm_causal,fixed_m_causal = ref_fixed_m_causal, old = old,realistic = realistic,prefix = prefix,use_gwash_m = False)
    ref_ldscores,ref_mu2_hat,ref_mu3_hat,X_ref = gen_ref_ldscores_panel(ref_data_gen,seed = seed_num,nPCs = nPCs, regress_X_on_PC = regress_X_on_PC, regress_y_on_PC = regress_y_on_PC)
    n_tilde = ref_data_gen.n
else:
    ref_data_gen = None
    ref_ldscores = None
    n_tilde = None

    ref_mu2_hat = None
    ref_mu3_hat = None
    X_ref = None

n_jobs = 1
    
my_data_gen = data_generation(n = n,m= m,Fst = Fst,rho1 = rho1,rho2 = rho2,sigma_s = sigma_s,h2_pop = h2_pop,pm_causal = pm_causal,fixed_m_causal = fixed_m_causal, old = old,model_Fst_in_realistic = model_Fst_in_realistic,realistic = realistic,prefix = prefix,ref_ldscores = ref_ldscores,ref_mu2_hat = ref_mu2_hat,ref_mu3_hat = ref_mu3_hat)
res_dfs = run_simulations(my_data_gen,seed = seed_num,scaleX = scaleX,scaley = scaley,num_sims = num_sims,nPCs = nPCs,regress_PC_out = regress_PC_out,regress_X_on_PC = regress_X_on_PC,regress_y_on_PC = regress_y_on_PC, multithreading = multithreading, n_jobs = n_jobs, realistic = realistic, calc_mu_hat_2_fast = calc_mu_hat_2_fast,reml_tol = reml_tol,track_progress = track_progress).run_single_simulation_publication(i=i)

res_dfs

In [ ]:
file = open('save_data/supplementary/realistic/single_param/pm_causal.pkl','rb')
saved_data = pickle.load(file)

saved_data_at_i = saved_data[str(pm_causal)].iloc[[i]].reset_index(drop = True)

In [ ]:
res_dfs - saved_data_at_i

### Reference Panel

In [ ]:
seed_num = 123

i = np.random.choice(np.arange(0,1000,1),size = 1).item()



n = 5000


X_properties = {'n': n,'m':10000,'pm_causal':None,'fixed_m_causal': False,'sigma_s':0, 'rho1':0.995,'rho2':None,'Fst':0}
ref_X_properties = {'ref_n': n,'ref_pm_causal':None,'ref_fixed_m_causal': False,'ref_sigma_s':0, 'ref_rho1':0.995,'ref_rho2':None,'ref_Fst':0}
ld_mat_properties = {'realistic': True,'model_Fst_in_realistic':True,'prefix': '1kg_p1_eur_ben/1kg_p1_eur_chr22','make_ref_ldscores':True}
method_properties = {'reml_tol': 1e-8,'reml_max_iters': 100}
stats_properties = {'scaleX': True,'scaley': True, 'nPCs':5,'regress_PC_out':False,'regress_X_on_PC': False,'regress_y_on_PC': False}
simul_properties = {'num_sims':np.nan,'h2_pop': 0.2}
debug_properties = {'calc_mu_hat_2_fast': True,'old':False,'track_progress':True}

list_of_dicts = [X_properties,ref_X_properties,ld_mat_properties,method_properties,stats_properties,simul_properties,debug_properties]
params  = combine_all_dicts(list_of_dicts)

to_run = dict()
to_run['demonstration'] = params

sim_key = 'demonstration'
locals().update(to_run[sim_key])
res_dict = dict()
res_dict_raw = dict()

#rhos = [0.995]
counter = -1
multithreading = True


counter += 1
pm_causals = [0.005,0.05,0.5,0.75,float(1)]
pm_causal = np.random.choice(pm_causals,size = 1).item()

if make_ref_ldscores:
    ref_data_gen = data_generation(n = ref_n,m= m,Fst = Fst,rho1 = ref_rho1,rho2 = ref_rho2,sigma_s = ref_sigma_s,h2_pop = h2_pop,pm_causal = pm_causal,fixed_m_causal = ref_fixed_m_causal, old = old,realistic = realistic,prefix = prefix,use_gwash_m = False)
    ref_ldscores,ref_mu2_hat,ref_mu3_hat,X_ref = gen_ref_ldscores_panel(ref_data_gen,seed = seed_num,nPCs = nPCs, regress_X_on_PC = regress_X_on_PC, regress_y_on_PC = regress_y_on_PC)
    n_tilde = ref_data_gen.n
else:
    ref_data_gen = None
    ref_ldscores = None
    n_tilde = None

    ref_mu2_hat = None
    ref_mu3_hat = None
    X_ref = None

n_jobs = 1
    
my_data_gen = data_generation(n = n,m= m,Fst = Fst,rho1 = rho1,rho2 = rho2,sigma_s = sigma_s,h2_pop = h2_pop,pm_causal = pm_causal,fixed_m_causal = fixed_m_causal, old = old,model_Fst_in_realistic = model_Fst_in_realistic,realistic = realistic,prefix = prefix,ref_ldscores = ref_ldscores,ref_mu2_hat = ref_mu2_hat,ref_mu3_hat = ref_mu3_hat)
res_dfs = run_simulations(my_data_gen,seed = seed_num,scaleX = scaleX,scaley = scaley,num_sims = num_sims,nPCs = nPCs,regress_PC_out = regress_PC_out,regress_X_on_PC = regress_X_on_PC,regress_y_on_PC = regress_y_on_PC, multithreading = multithreading, n_jobs = n_jobs, realistic = realistic, calc_mu_hat_2_fast = calc_mu_hat_2_fast,reml_tol = reml_tol,track_progress = track_progress).run_single_simulation_publication(i=i)

res_dfs

In [ ]:
file = open('save_data/main_text/realistic/single_param/pm_causal.pkl','rb')
saved_data = pickle.load(file)

saved_data_at_i = saved_data[str(pm_causal)].iloc[[i]].reset_index(drop = True)

In [ ]:
res_dfs - saved_data_at_i

## $\mathrm{F_{st}}$

### Individual-level

In [ ]:
seed_num = 123

i = np.random.choice(np.arange(0,1000,1),size = 1).item()

n = 5000

X_properties = {'n': n,'m':10000,'pm_causal':None,'fixed_m_causal': False,'sigma_s':0, 'rho1':0.995,'rho2':None,'Fst':0}
ref_X_properties = {'ref_n': n,'ref_pm_causal':None,'ref_fixed_m_causal': False,'ref_sigma_s':0, 'ref_rho1':0.995,'ref_rho2':None,'ref_Fst':0}
ld_mat_properties = {'realistic': True,'model_Fst_in_realistic':True,'prefix': '1kg_p1_eur_ben/1kg_p1_eur_chr22','make_ref_ldscores':False}
method_properties = {'reml_tol': 1e-8,'reml_max_iters': 100}
stats_properties = {'scaleX': True,'scaley': True, 'nPCs':5,'regress_PC_out':False,'regress_X_on_PC': False,'regress_y_on_PC': False}
simul_properties = {'num_sims':np.nan,'h2_pop': 0.2}
debug_properties = {'calc_mu_hat_2_fast': True,'old':False,'track_progress':True}

list_of_dicts = [X_properties,ref_X_properties,ld_mat_properties,method_properties,stats_properties,simul_properties,debug_properties]
params  = combine_all_dicts(list_of_dicts)

to_run = dict()
to_run['demonstration'] = params

sim_key = 'demonstration'
locals().update(to_run[sim_key])
res_dict = dict()
res_dict_raw = dict()

#rhos = [0.995]
counter = -1
multithreading = True


counter += 1
Fsts = [0,0.01,0.03,0.05,0.07,0.1]
Fst = np.random.choice(Fsts,size = 1).item()

if make_ref_ldscores:
    ref_data_gen = data_generation(n = ref_n,m= m,Fst = Fst,rho1 = ref_rho1,rho2 = ref_rho2,sigma_s = ref_sigma_s,h2_pop = h2_pop,pm_causal = pm_causal,fixed_m_causal = ref_fixed_m_causal, old = old,realistic = realistic,prefix = prefix,use_gwash_m = False)
    ref_ldscores,ref_mu2_hat,ref_mu3_hat,X_ref = gen_ref_ldscores_panel(ref_data_gen,seed = seed_num,nPCs = nPCs, regress_X_on_PC = regress_X_on_PC, regress_y_on_PC = regress_y_on_PC)
    n_tilde = ref_data_gen.n
else:
    ref_data_gen = None
    ref_ldscores = None
    n_tilde = None

    ref_mu2_hat = None
    ref_mu3_hat = None
    X_ref = None

n_jobs = 1
    
my_data_gen = data_generation(n = n,m= m,Fst = Fst,rho1 = rho1,rho2 = rho2,sigma_s = sigma_s,h2_pop = h2_pop,pm_causal = pm_causal,fixed_m_causal = fixed_m_causal, old = old,model_Fst_in_realistic = model_Fst_in_realistic,realistic = realistic,prefix = prefix,ref_ldscores = ref_ldscores,ref_mu2_hat = ref_mu2_hat,ref_mu3_hat = ref_mu3_hat)
res_dfs = run_simulations(my_data_gen,seed = seed_num,scaleX = scaleX,scaley = scaley,num_sims = num_sims,nPCs = nPCs,regress_PC_out = regress_PC_out,regress_X_on_PC = regress_X_on_PC,regress_y_on_PC = regress_y_on_PC, multithreading = multithreading, n_jobs = n_jobs, realistic = realistic, calc_mu_hat_2_fast = calc_mu_hat_2_fast,reml_tol = reml_tol,track_progress = track_progress).run_single_simulation_publication(i=i)

res_dfs

In [ ]:
file = open('save_data/supplementary/realistic/single_param/Fst.pkl','rb')
saved_data = pickle.load(file)

saved_data_at_i = saved_data[str(Fst)].iloc[[i]].reset_index(drop = True)

In [ ]:
res_dfs - saved_data_at_i

### Reference Panel

In [ ]:
seed_num = 123

i = np.random.choice(np.arange(0,1000,1),size = 1).item()

n = 5000

X_properties = {'n': n,'m':10000,'pm_causal':None,'fixed_m_causal': False,'sigma_s':0, 'rho1':0.995,'rho2':None,'Fst':0}
ref_X_properties = {'ref_n': n,'ref_pm_causal':None,'ref_fixed_m_causal': False,'ref_sigma_s':0, 'ref_rho1':0.995,'ref_rho2':None,'ref_Fst':0}
ld_mat_properties = {'realistic': True,'model_Fst_in_realistic':True,'prefix': '1kg_p1_eur_ben/1kg_p1_eur_chr22','make_ref_ldscores':True}
method_properties = {'reml_tol': 1e-8,'reml_max_iters': 100}
stats_properties = {'scaleX': True,'scaley': True, 'nPCs':5,'regress_PC_out':False,'regress_X_on_PC': False,'regress_y_on_PC': False}
simul_properties = {'num_sims':np.nan,'h2_pop': 0.2}
debug_properties = {'calc_mu_hat_2_fast': True,'old':False,'track_progress':True}

list_of_dicts = [X_properties,ref_X_properties,ld_mat_properties,method_properties,stats_properties,simul_properties,debug_properties]
params  = combine_all_dicts(list_of_dicts)

to_run = dict()
to_run['demonstration'] = params

sim_key = 'demonstration'
locals().update(to_run[sim_key])
res_dict = dict()
res_dict_raw = dict()

#rhos = [0.995]
counter = -1
multithreading = True


counter += 1
Fsts = [0,0.01,0.03,0.05,0.07,0.1]
Fst = np.random.choice(Fsts,size = 1).item()

if make_ref_ldscores:
    ref_data_gen = data_generation(n = ref_n,m= m,Fst = Fst,rho1 = ref_rho1,rho2 = ref_rho2,sigma_s = ref_sigma_s,h2_pop = h2_pop,pm_causal = pm_causal,fixed_m_causal = ref_fixed_m_causal, old = old,realistic = realistic,prefix = prefix,use_gwash_m = False)
    ref_ldscores,ref_mu2_hat,ref_mu3_hat,X_ref = gen_ref_ldscores_panel(ref_data_gen,seed = seed_num,nPCs = nPCs, regress_X_on_PC = regress_X_on_PC, regress_y_on_PC = regress_y_on_PC)
    n_tilde = ref_data_gen.n
else:
    ref_data_gen = None
    ref_ldscores = None
    n_tilde = None

    ref_mu2_hat = None
    ref_mu3_hat = None
    X_ref = None

n_jobs = 1
    
my_data_gen = data_generation(n = n,m= m,Fst = Fst,rho1 = rho1,rho2 = rho2,sigma_s = sigma_s,h2_pop = h2_pop,pm_causal = pm_causal,fixed_m_causal = fixed_m_causal, old = old,model_Fst_in_realistic = model_Fst_in_realistic,realistic = realistic,prefix = prefix,ref_ldscores = ref_ldscores,ref_mu2_hat = ref_mu2_hat,ref_mu3_hat = ref_mu3_hat)
res_dfs = run_simulations(my_data_gen,seed = seed_num,scaleX = scaleX,scaley = scaley,num_sims = num_sims,nPCs = nPCs,regress_PC_out = regress_PC_out,regress_X_on_PC = regress_X_on_PC,regress_y_on_PC = regress_y_on_PC, multithreading = multithreading, n_jobs = n_jobs, realistic = realistic, calc_mu_hat_2_fast = calc_mu_hat_2_fast,reml_tol = reml_tol,track_progress = track_progress).run_single_simulation_publication(i=i)

res_dfs

In [ ]:
file = open('save_data/main_text/realistic/single_param/Fst.pkl','rb')
saved_data = pickle.load(file)

saved_data_at_i = saved_data[str(Fst)].iloc[[i]].reset_index(drop = True)

In [ ]:
res_dfs - saved_data_at_i

## $\sigma_{s}$

### Individual-level

In [ ]:
seed_num = 123

i = np.random.choice(np.arange(0,1000,1),size = 1).item()

n = 5000

X_properties = {'n': n,'m':10000,'pm_causal':None,'fixed_m_causal': False,'sigma_s':0, 'rho1':0.995,'rho2':None,'Fst':0}
ref_X_properties = {'ref_n': n,'ref_pm_causal':None,'ref_fixed_m_causal': False,'ref_sigma_s':0, 'ref_rho1':0.995,'ref_rho2':None,'ref_Fst':0}
ld_mat_properties = {'realistic': True,'model_Fst_in_realistic':True,'prefix': '1kg_p1_eur_ben/1kg_p1_eur_chr22','make_ref_ldscores':False}
method_properties = {'reml_tol': 1e-8,'reml_max_iters': 100}
stats_properties = {'scaleX': True,'scaley': True, 'nPCs':5,'regress_PC_out':False,'regress_X_on_PC': False,'regress_y_on_PC': False}
simul_properties = {'num_sims':np.nan,'h2_pop': 0.2}
debug_properties = {'calc_mu_hat_2_fast': True,'old':False,'track_progress':True}

list_of_dicts = [X_properties,ref_X_properties,ld_mat_properties,method_properties,stats_properties,simul_properties,debug_properties]
params  = combine_all_dicts(list_of_dicts)

to_run = dict()
to_run['demonstration'] = params

sim_key = 'demonstration'
locals().update(to_run[sim_key])
res_dict = dict()
res_dict_raw = dict()

#rhos = [0.995]
counter = -1
multithreading = True


counter += 1
sigma_ss = [0,0.2,0.4,0.6,0.8]
sigma_s = np.random.choice(sigma_ss,size = 1).item()

if make_ref_ldscores:
    ref_data_gen = data_generation(n = ref_n,m= m,Fst = Fst,rho1 = ref_rho1,rho2 = ref_rho2,sigma_s = sigma_s,h2_pop = h2_pop,pm_causal = pm_causal,fixed_m_causal = ref_fixed_m_causal, old = old,realistic = realistic,prefix = prefix,use_gwash_m = False)
    ref_ldscores,ref_mu2_hat,ref_mu3_hat,X_ref = gen_ref_ldscores_panel(ref_data_gen,seed = seed_num,nPCs = nPCs, regress_X_on_PC = regress_X_on_PC, regress_y_on_PC = regress_y_on_PC)
    n_tilde = ref_data_gen.n
else:
    ref_data_gen = None
    ref_ldscores = None
    n_tilde = None

    ref_mu2_hat = None
    ref_mu3_hat = None
    X_ref = None

n_jobs = 1
    
my_data_gen = data_generation(n = n,m= m,Fst = Fst,rho1 = rho1,rho2 = rho2,sigma_s = sigma_s,h2_pop = h2_pop,pm_causal = pm_causal,fixed_m_causal = fixed_m_causal, old = old,model_Fst_in_realistic = model_Fst_in_realistic,realistic = realistic,prefix = prefix,ref_ldscores = ref_ldscores,ref_mu2_hat = ref_mu2_hat,ref_mu3_hat = ref_mu3_hat)
res_dfs = run_simulations(my_data_gen,seed = seed_num,scaleX = scaleX,scaley = scaley,num_sims = num_sims,nPCs = nPCs,regress_PC_out = regress_PC_out,regress_X_on_PC = regress_X_on_PC,regress_y_on_PC = regress_y_on_PC, multithreading = multithreading, n_jobs = n_jobs, realistic = realistic, calc_mu_hat_2_fast = calc_mu_hat_2_fast,reml_tol = reml_tol,track_progress = track_progress).run_single_simulation_publication(i=i)

res_dfs

In [ ]:
file = open('save_data/supplementary/realistic/single_param/sigma_s.pkl','rb')
saved_data = pickle.load(file)

saved_data_at_i = saved_data[str(sigma_s)].iloc[[i]].reset_index(drop = True)

In [ ]:
res_dfs - saved_data_at_i

### Reference Panel

In [ ]:
seed_num = 123

i = np.random.choice(np.arange(0,1000,1),size = 1).item()

n = 5000

X_properties = {'n': n,'m':10000,'pm_causal':None,'fixed_m_causal': False,'sigma_s':0, 'rho1':0.995,'rho2':None,'Fst':0}
ref_X_properties = {'ref_n': n,'ref_pm_causal':None,'ref_fixed_m_causal': False,'ref_sigma_s':0, 'ref_rho1':0.995,'ref_rho2':None,'ref_Fst':0}
ld_mat_properties = {'realistic': True,'model_Fst_in_realistic':True,'prefix':'1kg_p1_eur_ben/1kg_p1_eur_chr22','make_ref_ldscores':True}
method_properties = {'reml_tol': 1e-8,'reml_max_iters': 100}
stats_properties = {'scaleX': True,'scaley': True, 'nPCs':5,'regress_PC_out':False,'regress_X_on_PC': False,'regress_y_on_PC': False}
simul_properties = {'num_sims':np.nan,'h2_pop': 0.2}
debug_properties = {'calc_mu_hat_2_fast': True,'old':False,'track_progress':True}

list_of_dicts = [X_properties,ref_X_properties,ld_mat_properties,method_properties,stats_properties,simul_properties,debug_properties]
params  = combine_all_dicts(list_of_dicts)

to_run = dict()
to_run['demonstration'] = params

sim_key = 'demonstration'
locals().update(to_run[sim_key])
res_dict = dict()
res_dict_raw = dict()

#rhos = [0.995]
counter = -1
multithreading = True


counter += 1
sigma_ss = [0,0.2,0.4,0.6,0.8]
sigma_s = np.random.choice(sigma_ss,size = 1).item()

if make_ref_ldscores:
    ref_data_gen = data_generation(n = ref_n,m= m,Fst = Fst,rho1 = ref_rho1,rho2 = ref_rho2,sigma_s = sigma_s,h2_pop = h2_pop,pm_causal = pm_causal,fixed_m_causal = ref_fixed_m_causal, old = old,realistic = realistic,prefix = prefix,use_gwash_m = False)
    ref_ldscores,ref_mu2_hat,ref_mu3_hat,X_ref = gen_ref_ldscores_panel(ref_data_gen,seed = seed_num,nPCs = nPCs, regress_X_on_PC = regress_X_on_PC, regress_y_on_PC = regress_y_on_PC)
    n_tilde = ref_data_gen.n
else:
    ref_data_gen = None
    ref_ldscores = None
    n_tilde = None

    ref_mu2_hat = None
    ref_mu3_hat = None
    X_ref = None

n_jobs = 1
    
my_data_gen = data_generation(n = n,m= m,Fst = Fst,rho1 = rho1,rho2 = rho2,sigma_s = sigma_s,h2_pop = h2_pop,pm_causal = pm_causal,fixed_m_causal = fixed_m_causal, old = old,model_Fst_in_realistic = model_Fst_in_realistic,realistic = realistic,prefix = prefix,ref_ldscores = ref_ldscores,ref_mu2_hat = ref_mu2_hat,ref_mu3_hat = ref_mu3_hat)
res_dfs = run_simulations(my_data_gen,seed = seed_num,scaleX = scaleX,scaley = scaley,num_sims = num_sims,nPCs = nPCs,regress_PC_out = regress_PC_out,regress_X_on_PC = regress_X_on_PC,regress_y_on_PC = regress_y_on_PC, multithreading = multithreading, n_jobs = n_jobs, realistic = realistic, calc_mu_hat_2_fast = calc_mu_hat_2_fast,reml_tol = reml_tol,track_progress = track_progress).run_single_simulation_publication(i=i)

res_dfs

In [ ]:
file = open('save_data/main_text/realistic/single_param/sigma_s.pkl','rb')
saved_data = pickle.load(file)

if sigma_s == 0:
    sigma_s = int(sigma_s)

saved_data_at_i = saved_data[str(sigma_s)].iloc[[i]].reset_index(drop = True)

In [ ]:
res_dfs - saved_data_at_i

## $\mathrm{F_{st}} + \sigma_{s}$

### Individual-level

In [ ]:
seed_num = 123

i = np.random.choice(np.arange(0,1000,1),size = 1).item()



n,m = 5000,10000


X_properties = {'n': n,'m':10000,'pm_causal':None,'fixed_m_causal': False,'sigma_s':0, 'rho1':0.995,'rho2':None,'Fst':0}
ref_X_properties = {'ref_n': n,'ref_pm_causal':None,'ref_fixed_m_causal': False,'ref_sigma_s':0, 'ref_rho1':0.995,'ref_rho2':None,'ref_Fst':0}
ld_mat_properties = {'realistic': True,'model_Fst_in_realistic':True,'prefix': '1kg_p1_eur_ben/1kg_p1_eur_chr22','make_ref_ldscores':False}
method_properties = {'reml_tol': 1e-8,'reml_max_iters': 100}
stats_properties = {'scaleX': True,'scaley': True, 'nPCs':5,'regress_PC_out':False,'regress_X_on_PC': False,'regress_y_on_PC': False}
simul_properties = {'num_sims':np.nan,'h2_pop': 0.2}
debug_properties = {'calc_mu_hat_2_fast': True,'old':False,'track_progress':True}

list_of_dicts = [X_properties,ref_X_properties,ld_mat_properties,method_properties,stats_properties,simul_properties,debug_properties]
params  = combine_all_dicts(list_of_dicts)

to_run = dict()
to_run['demonstration'] = params

sim_key = 'demonstration'
locals().update(to_run[sim_key])
res_dict = dict()
res_dict_raw = dict()

#rhos = [0.995]
counter = -1
multithreading = True


counter += 1
Fsts = [0.05,0.1]
Fst = np.random.choice(Fsts,size = 1).item()
sigma_ss = [0,0.2,0.4,0.6,0.8]
sigma_s = np.random.choice(sigma_ss,size = 1).item()

if make_ref_ldscores:
    ref_data_gen = data_generation(n = ref_n,m= m,Fst = Fst,rho1 = ref_rho1,rho2 = ref_rho2,sigma_s = sigma_s,h2_pop = h2_pop,pm_causal = pm_causal,fixed_m_causal = ref_fixed_m_causal, old = old,realistic = realistic,prefix = prefix,use_gwash_m = False)
    ref_ldscores,ref_mu2_hat,ref_mu3_hat,X_ref = gen_ref_ldscores_panel(ref_data_gen,seed = seed_num,nPCs = nPCs, regress_X_on_PC = regress_X_on_PC, regress_y_on_PC = regress_y_on_PC)
    n_tilde = ref_data_gen.n
else:
    ref_data_gen = None
    ref_ldscores = None
    n_tilde = None

    ref_mu2_hat = None
    ref_mu3_hat = None
    X_ref = None

n_jobs = 1
    
my_data_gen = data_generation(n = n,m= m,Fst = Fst,rho1 = rho1,rho2 = rho2,sigma_s = sigma_s,h2_pop = h2_pop,pm_causal = pm_causal,fixed_m_causal = fixed_m_causal, old = old,realistic = realistic,prefix = prefix,ref_ldscores = ref_ldscores,ref_mu2_hat = ref_mu2_hat,ref_mu3_hat = ref_mu3_hat)
res_dfs = run_simulations(my_data_gen,seed = seed_num,scaleX = scaleX,scaley = scaley,num_sims = num_sims,nPCs = nPCs,regress_PC_out = regress_PC_out,regress_X_on_PC = regress_X_on_PC,regress_y_on_PC = regress_y_on_PC, multithreading = multithreading, n_jobs = n_jobs, realistic = realistic, calc_mu_hat_2_fast = calc_mu_hat_2_fast,reml_tol = reml_tol,track_progress = track_progress).run_single_simulation_publication(i=i)

res_dfs

In [ ]:
file = open('save_data/supplementary/realistic/double_param/Fst'+str(Fst).replace('.','')+'sigma_s.pkl','rb')
saved_data = pickle.load(file)

if sigma_s == 0:
    sigma_s = int(sigma_s)

saved_data_at_i = saved_data[str(sigma_s)].iloc[[i]].reset_index(drop = True)

In [ ]:
res_dfs - saved_data_at_i

### Reference Panel

In [ ]:
seed_num = 123

i = np.random.choice(np.arange(0,1000,1),size = 1).item()



n,m = 5000,10000


X_properties = {'n': n,'m':10000,'pm_causal':None,'fixed_m_causal': False,'sigma_s':0, 'rho1':0.995,'rho2':None,'Fst':0}
ref_X_properties = {'ref_n': n,'ref_pm_causal':None,'ref_fixed_m_causal': False,'ref_sigma_s':0, 'ref_rho1':0.995,'ref_rho2':None,'ref_Fst':0}
ld_mat_properties = {'realistic': True,'model_Fst_in_realistic':True,'prefix': '1kg_p1_eur_ben/1kg_p1_eur_chr22','make_ref_ldscores':True}
method_properties = {'reml_tol': 1e-8,'reml_max_iters': 100}
stats_properties = {'scaleX': True,'scaley': True, 'nPCs':5,'regress_PC_out':False,'regress_X_on_PC': False,'regress_y_on_PC': False}
simul_properties = {'num_sims':np.nan,'h2_pop': 0.2}
debug_properties = {'calc_mu_hat_2_fast': True,'old':False,'track_progress':True}

list_of_dicts = [X_properties,ref_X_properties,ld_mat_properties,method_properties,stats_properties,simul_properties,debug_properties]
params  = combine_all_dicts(list_of_dicts)

to_run = dict()
to_run['demonstration'] = params

sim_key = 'demonstration'
locals().update(to_run[sim_key])
res_dict = dict()
res_dict_raw = dict()

#rhos = [0.995]
counter = -1
multithreading = True


counter += 1
Fsts = [0.05,0.1]
Fst = np.random.choice(Fsts,size = 1).item()
sigma_ss = [0,0.2,0.4,0.6,0.8]
sigma_s = np.random.choice(sigma_ss,size = 1).item()

if make_ref_ldscores:
    ref_data_gen = data_generation(n = ref_n,m= m,Fst = Fst,rho1 = ref_rho1,rho2 = ref_rho2,sigma_s = sigma_s,h2_pop = h2_pop,pm_causal = pm_causal,fixed_m_causal = ref_fixed_m_causal, old = old,realistic = realistic,prefix = prefix,use_gwash_m = False)
    ref_ldscores,ref_mu2_hat,ref_mu3_hat,X_ref = gen_ref_ldscores_panel(ref_data_gen,seed = seed_num,nPCs = nPCs, regress_X_on_PC = regress_X_on_PC, regress_y_on_PC = regress_y_on_PC)
    n_tilde = ref_data_gen.n
else:
    ref_data_gen = None
    ref_ldscores = None
    n_tilde = None

    ref_mu2_hat = None
    ref_mu3_hat = None
    X_ref = None

n_jobs = 1
    
my_data_gen = data_generation(n = n,m= m,Fst = Fst,rho1 = rho1,rho2 = rho2,sigma_s = sigma_s,h2_pop = h2_pop,pm_causal = pm_causal,fixed_m_causal = fixed_m_causal, old = old,realistic = realistic,prefix = prefix,ref_ldscores = ref_ldscores,ref_mu2_hat = ref_mu2_hat,ref_mu3_hat = ref_mu3_hat)
res_dfs = run_simulations(my_data_gen,seed = seed_num,scaleX = scaleX,scaley = scaley,num_sims = num_sims,nPCs = nPCs,regress_PC_out = regress_PC_out,regress_X_on_PC = regress_X_on_PC,regress_y_on_PC = regress_y_on_PC, multithreading = multithreading, n_jobs = n_jobs, realistic = realistic, calc_mu_hat_2_fast = calc_mu_hat_2_fast,reml_tol = reml_tol,track_progress = track_progress).run_single_simulation_publication(i=i)

res_dfs

In [ ]:
file = open('save_data/main_text/realistic/double_param/Fst'+str(Fst).replace('.','')+'sigma_s.pkl','rb')
saved_data = pickle.load(file)

if sigma_s == 0:
    sigma_s = int(sigma_s)

saved_data_at_i = saved_data[str(sigma_s)].iloc[[i]].reset_index(drop = True)

In [ ]:
res_dfs - saved_data_at_i